In [16]:
%load_ext autoreload
%autoreload 2

import sklearn
import scipy 
import numpy as np
import pandas as pd
import os
import sys
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
sys.path.append(os.path.abspath("src"))
import jdcoot
import math
from scipy.io import loadmat

from jdcoot.models.discrete_unsupervised_jdcoot import discrete_unsupervised_jdcoot
from jdcoot.models.discrete_semisupervised_jdcoot import discrete_semisupervised_jdcoot
from jdcoot.models.discrete_partial_jdcoot import discrete_partial_jdcoot
from jdcoot.models.discrete_unsupervised_coot import discrete_unsupervised_coot
from jdcoot.models.discrete_semisupervised_coot import discrete_semisupervised_coot
from jdcoot.models.discrete_partial_coot import discrete_partial_coot
from jdcoot.models.discrete_semisupervised_reference import discrete_semisupervised_reference
from jdcoot.models.discrete_partial_reference import discrete_partial_reference
from jdcoot.utils import xcolumns, continuous_classifiers, continuous_accuracy
from sklearn.model_selection import train_test_split

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

# les donnees caffeNet GoogleNet

In [3]:
featuresToUse = ["CaffeNet4096", "GoogleNet1024"] 
#featuresToUse = ["CaffeNet4096", "CaffeNet4096"] 
sourceDomainName = ['caltech10'] #['caltech10','amazon','webcam']
targetDomainName = ['amazon'] #['caltech10','amazon','webcam']

min_max_scaler = sklearn.preprocessing.MinMaxScaler()
# Collab
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[0],
                                                 "caltech10" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
S_data = [feat, labels]
S_nClass = len(np.unique(labels)) # nb de class in source data
possible_data = loadmat(os.path.join("data/", "features", featuresToUse[1],
                                                 "amazon" + '.mat'))
feat = possible_data['fts'].astype(float)
labels = possible_data['labels'].ravel()
T_data = [feat, labels]
T_nClass = len(np.unique(labels)) # nb de class in target data

source=pd.DataFrame(np.concatenate((S_data[0],S_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(S_data[0].shape[1])]+['Z'])
target=pd.DataFrame(np.concatenate((T_data[0],T_data[1].reshape(-1,1)),axis=1),columns=['X'+str(i) for i in range(T_data[0].shape[1])]+['Z'])

source.loc[:,'Z'] = source.loc[:, 'Z'] - 1
target.loc[:,'Z'] = target.loc[:, 'Z'] - 1


In [4]:
results = []
numRepetitions = 10
alpha =1.5# hyperparamètre devant la loss a été optimé
prop_target = 0.005
algo = "sinkhorn"
reg = 1


In [5]:
a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
S = source.iloc[a, :].reset_index(drop=True)
T = target.iloc[b, :].reset_index(drop=True)

In [7]:
pure_source, pure_target, test_source, test_target = discrete_unsupervised_jdcoot(
                S, T, S_test, T_test,algo="sinkhorn",reg=1,batch_size=20,alpha =1.5
            )
test_target

Delta: 0.02092798685283678 	  Loss: 2.1473654086839193 	 Accuracy: 0.20860495436766624
Delta: 0.019073847854404635 	  Loss: 2.048302812754046 	 Accuracy: 0.37157757496740546
Delta: 0.01676452405183621 	  Loss: 1.8677940006397904 	 Accuracy: 0.4876140808344198
Delta: 0.014049839000854727 	  Loss: 1.7163144169832814 	 Accuracy: 0.41851368970013036
Delta: 0.01203145438126765 	  Loss: 1.6360994358998124 	 Accuracy: 0.39374185136897
Delta: 0.010175359826428095 	  Loss: 1.5970084424659972 	 Accuracy: 0.3833116036505867
Delta: 0.008718547889898237 	  Loss: 1.5790902327283272 	 Accuracy: 0.3741851368970013
Delta: 0.008127365247757617 	  Loss: 1.5700033860280627 	 Accuracy: 0.36766623207301175
Delta: 0.007497062249503367 	  Loss: 1.5649376118408074 	 Accuracy: 0.3376792698826597


/home/vgares/mambaforge/envs/jdcoot/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:666: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


Delta: 0.007013439020467204 	  Loss: 1.5619345604898287 	 Accuracy: 0.3363754889178618
Delta: 0.006765096592841762 	  Loss: 1.559713940518212 	 Accuracy: 0.3376792698826597
Delta: 0.00524886880493173 	  Loss: 1.5585495273837306 	 Accuracy: 0.3376792698826597
Delta: 0.004513107968689477 	  Loss: 1.5577329387462076 	 Accuracy: 0.3376792698826597
Delta: 0.004889221528666393 	  Loss: 1.557083600310453 	 Accuracy: 0.3376792698826597
Delta: 0.0044407014412937455 	  Loss: 1.5567998672979082 	 Accuracy: 0.3376792698826597
Delta: 0.004125334229077259 	  Loss: 1.55655295715381 	 Accuracy: 0.34028683181225555
Delta: 0.004656910069616502 	  Loss: 1.5563580600682725 	 Accuracy: 0.3389830508474576
Delta: 0.0022984233248476275 	  Loss: 1.5562576866919726 	 Accuracy: 0.3389830508474576
Delta: 0.003965730686381726 	  Loss: 1.5560767563588846 	 Accuracy: 0.34028683181225555
Delta: 0.0012283781359316453 	  Loss: 1.5560912717845174 	 Accuracy: 0.34028683181225555
Delta: 0.001000441287283334 	  Loss: 1.556

np.float64(0.3089005235602094)

In [7]:
pure_source, pure_target, test_source, test_target = discrete_unsupervised_coot(
                S, T, S_test, T_test,algo="sinkhorn",reg=1,batch_size=20
            )
test_target

Delta:       0.0156335 	 Loss:       2.2046902
Delta:       0.0209275 	 Loss:       2.1683288
Delta:       0.0187107 	 Loss:       2.1067086
Delta:       0.0139696 	 Loss:       2.0724738
Delta:       0.0105210 	 Loss:       2.0659279
Delta:       0.0099668 	 Loss:       2.0633529
Delta:       0.0097964 	 Loss:       2.0600302
Delta:       0.0083934 	 Loss:       2.0574664
Delta:       0.0069205 	 Loss:       2.0561957
Delta:       0.0060637 	 Loss:       2.0554801
Delta:       0.0038115 	 Loss:       2.0552763
Delta:       0.0029421 	 Loss:       2.0551820
Delta:       0.0025912 	 Loss:       2.0551839
Delta:       0.0021900 	 Loss:       2.0551567
Delta:       0.0026604 	 Loss:       2.0551325
Delta:       0.0010400 	 Loss:       2.0551285
Delta:       0.0018294 	 Loss:       2.0551359


KeyboardInterrupt: 

In [7]:
n_target = len(T.Z)
l_train, l_test = train_test_split(
        np.arange(n_target),
        train_size=0.01,
    )

In [9]:
pure_source, pure_target, test_source, test_target = discrete_semisupervised_jdcoot(
                S, T, S_test, T_test,l_train, l_test,algo="sinkhorn",reg=1,batch_size=20,alpha =0         
            )
test_target

KeyboardInterrupt: 

In [10]:
pure_source, pure_target, test_source, test_target = discrete_semisupervised_coot(
                S, T, S_test, T_test,l_train, l_test,algo="sinkhorn",reg=1,batch_size=20            
            )
test_target

Delta:       0.0159551 	 Loss:       2.1893512
Delta:       0.0201767 	 Loss:       2.1314885


KeyboardInterrupt: 

In [11]:
pure_source, pure_target, test_source, test_target = discrete_semisupervised_reference(
                S, T, S_test, T_test,l_train, l_test,algo="sinkhorn",reg=1,batch_size=20                      
            )
test_target

np.float64(0.4039473684210526)

In [12]:
prop_source=0.5
prop_target=0.005 
n_target = len(T.Z)
n_source = len(S.Z)
l_source_train, l_source_test = train_test_split(
        np.arange(n_source),
        train_size=prop_source
    )

l_target_train, l_target_test = train_test_split(
        np.arange(n_target),
        train_size=prop_target
    )

In [13]:
pure_source, pure_target, test_source, test_target = discrete_partial_jdcoot(
                S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test,algo="sinkhorn",reg=1,batch_size=20,alpha =1.5              
            )
test_target

Delta: 0.01784830223793116 	  Loss: 1.9583492060041268 	 Accuracy: 0.28272251308900526
Delta: 0.017081034324712 	  Loss: 1.9541692425028412 	 Accuracy: 0.3010471204188482
Delta: 0.013936168639738327 	  Loss: 1.9414185778641526 	 Accuracy: 0.31675392670157065


KeyboardInterrupt: 

In [17]:
pure_source, pure_target, test_source, test_target = discrete_partial_reference(
                S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test,algo=algo,reg=reg,
                prop_source=0.95,
                prop_target=0.005
            )
test_target


np.float64(0.3193717277486911)

In [ ]:
import numpy as np
# Exemple de valeurs à tester pour alpha
alpha_values = np.linspace(1, 3, 6)  # 0, 0.1, 0.2, ..., 1.0
best_alpha = None
best_score = -np.inf  # ou 0 selon ta métrique
results = []
numRepetitions = 1
for repe in range(numRepetitions):
   
        a = np.random.choice(np.arange(len(source)), math.ceil(0.8 * len(source)), replace=False)
        b = np.random.choice(np.arange(len(target)), math.ceil(0.8 * len(target)), replace=False)

        S_test = source.iloc[np.setdiff1d(np.arange(len(source)), a), :].reset_index(drop=True)
        T_test = target.iloc[np.setdiff1d(np.arange(len(target)), b), :].reset_index(drop=True)
        S = source.iloc[a, :].reset_index(drop=True)
        T = target.iloc[b, :].reset_index(drop=True)
        n_target = len(T.Z)
        n_source = len(S.Z)
        l_source_train, l_source_test = train_test_split(
            np.arange(n_source),
            train_size=prop_source
        )

        l_target_train, l_target_test = train_test_split(
            np.arange(n_target),
            train_size=prop_target
            )
        for a in alpha_values:
            pure_source, pure_target, test_source, test_target = \
            discrete_partial_jdcoot(S, T, S_test, T_test,l_source_train, l_source_test,l_target_train, l_target_test,algo="sinkhorn",reg=1,batch_size=20,alpha =a)

            score = test_target  
    
            if score > best_score:
                best_score = score
                best_alpha = a

        results.append({
         "repetition": repe,
         "recoding": "jdcoot",
         "learning": "unsupervised",
         "alpha": best_alpha,
     })

   

df_results = pd.DataFrame(results)


df_results

df_summary = (
    df_results
    .groupby(
        ["recoding", "learning", "alpha"],
        as_index=False
    )
    .agg(
        alpha_mean=("alpha", "mean"),
    )
)

df_summary

df_summary.to_excel("results_mean.xlsx", index=False)


Delta: 0.017223660338058226 	  Loss: 2.0444923080506943 	 Accuracy: 0.18586387434554974
Delta: 0.018464328934951865 	  Loss: 2.0445552692213846 	 Accuracy: 0.2212041884816754


In [6]:
df_summary.to_excel("results_mean.xlsx", index=False)

In [8]:
df_results 

,repetition,recoding,learning,alpha
0,0,jdcoot,unsupervised,1.9
1,1,jdcoot,unsupervised,1.9
2,2,jdcoot,unsupervised,1.9
3,3,jdcoot,unsupervised,1.9
4,4,jdcoot,unsupervised,1.9
5,5,jdcoot,unsupervised,1.9
6,6,jdcoot,unsupervised,1.9
7,7,jdcoot,unsupervised,1.0
8,8,jdcoot,unsupervised,1.0
9,9,jdcoot,unsupervised,1.0
